In [ ]:
import pandas as pd
import numpy as np

import os
from tqdm import tqdm
import zipfile


import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import librosa
import soundfile as sf
from scipy.fft import dct

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


**Checking Files structure**


In [ ]:
from genericpath import isdir
base_path='/content/drive/MyDrive/ForensicAudio'
if os.path.exists(base_path):
  print("found path")
for t in os.listdir(base_path):
  print(t)
protocol_path=os.path.join(base_path,"ASVspoof2019.LA.cm.train.trn.txt")
print(f"protocol found:{os.path.exists(protocol_path)}")
Audio_path=os.path.join(base_path,"ASVspoof2019_LA_train/flac")
print(f"Audio file found:{os.path.exists(Audio_path)}")
flac_files=[f for f in os.listdir(Audio_path) if f.endswith('.flac')]
print(f"FLAC files found: {len(flac_files)}")
print(f"Sample files: {flac_files[:3]}")



found path
ASVspoof2019.LA.cm.train.trn.txt
ASVspoof2019_LA_train
features
protocol found:True
Audio file found:True
FLAC files found: 25380
Sample files: ['LA_T_5250203.flac', 'LA_T_1576430.flac', 'LA_T_5337549.flac']


**Read Protocol file:**

In [ ]:
df=pd.read_csv(protocol_path,
               sep=' ',
               header=None,
               names=['speaker_id', 'file_id', 'system_id', 'key', 'label'])
df['filepath']=df['file_id'].apply(lambda x: os.path.join(Audio_path,x+'.flac'))
df['binary_label']=(df['label']=='spoof').astype(int)
print(df.head(10))
print(f"labels distribution:{df['label'].value_counts()}")
print(f"\nAttack types:\n{df['system_id'].value_counts()}")

  speaker_id       file_id system_id key     label  \
0    LA_0079  LA_T_1138215         -   -  bonafide   
1    LA_0079  LA_T_1271820         -   -  bonafide   
2    LA_0079  LA_T_1272637         -   -  bonafide   
3    LA_0079  LA_T_1276960         -   -  bonafide   
4    LA_0079  LA_T_1341447         -   -  bonafide   
5    LA_0079  LA_T_1363611         -   -  bonafide   
6    LA_0079  LA_T_1596451         -   -  bonafide   
7    LA_0079  LA_T_1608170         -   -  bonafide   
8    LA_0079  LA_T_1684951         -   -  bonafide   
9    LA_0079  LA_T_1699801         -   -  bonafide   

                                            filepath  binary_label  
0  /content/drive/MyDrive/ForensicAudio/ASVspoof2...             0  
1  /content/drive/MyDrive/ForensicAudio/ASVspoof2...             0  
2  /content/drive/MyDrive/ForensicAudio/ASVspoof2...             0  
3  /content/drive/MyDrive/ForensicAudio/ASVspoof2...             0  
4  /content/drive/MyDrive/ForensicAudio/ASVspoof2...        

**Fix class weights**

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
classes=np.array([0,1])
weights=compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=df['binary_label'].values
)
class_weights=torch.FloatTensor(weights)
print(f"Class weights → bonafide: {weights[0]:.2f} | spoof: {weights[1]:.2f}")

Class weights → bonafide: 4.92 | spoof: 0.56


In [ ]:
for label in ['bonafide', 'spoof']:
    sample = df[df['label'] == label].iloc[0]
    y, sr = librosa.load(sample['filepath'], sr=16000)
    print(f"\n{label.upper()}")
    print(f"  File    : {sample['file_id']}")
    print(f"  Duration: {len(y)/sr:.2f} sec")
    print(f"  Shape   : {y.shape}")
    print(f"  SR      : {sr} Hz")


BONAFIDE
  File    : LA_T_1138215
  Duration: 3.46 sec
  Shape   : (55329,)
  SR      : 16000 Hz

SPOOF
  File    : LA_T_1004644
  Duration: 1.92 sec
  Shape   : (30769,)
  SR      : 16000 Hz


**LFCC Extration:one file**

In [ ]:
def lfcc_extraction(filepath,n_lfcc=20,n_filter=70, n_fft=512,win_length=400, hop_length=160, sr=16000, max_frames=300):
  y,_=librosa.load(filepath, sr=sr,mono=True)
  target_len=sr*3 # we choose to work with a constant audio-duratiion of 3
  if len(y)<target_len:
    y=np.pad(y,(0,target_len-len(y)))
  else:
    y=y[:target_len]
  #Pre-Emphasis Filter: Boost heigh frequencies before analysis:
  y=np.append(y[0],y[1:]-0.97*y[:-1])
  #power spectrum:
  S=np.abs(librosa.stft(y,n_fft=n_fft,hop_length=hop_length,win_length=win_length)) ** 2

  #linear filterbank:
  freq_bins  = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
  linear_fb=np.zeros((n_filter,len(freq_bins)))
  freq_points=np.linspace(0,sr/2,n_filter+2)
  for i in range(n_filter):
        start  = freq_points[i]
        center = freq_points[i+1]
        end    = freq_points[i+2]
        for j, f in enumerate(freq_bins):
            if start <= f <= center:
                linear_fb[i,j] = (f - start) / (center - start + 1e-8)
            elif center < f <= end:
                linear_fb[i,j] = (end - f) / (end - center + 1e-8)
  # Log filterbank energies → DCT → LFCC
  filter_energies = np.dot(linear_fb, S)
  log_energies    = np.log(filter_energies + 1e-8)
  lfcc = dct(log_energies, type=2, axis=0)[:n_lfcc]

  # Fix time dimension
  if lfcc.shape[1] < max_frames:
      lfcc = np.pad(lfcc, ((0,0),(0, max_frames - lfcc.shape[1])))
  else:
        lfcc = lfcc[:, :max_frames]

  return lfcc


In [ ]:
for label in ['bonafide', 'spoof']:
    sample = df[df['label'] == label].iloc[0]
    lfcc = lfcc_extraction(sample['filepath'])
    print(f"{label.upper()} → LFCC shape: {lfcc.shape} | min: {lfcc.min():.2f} | max: {lfcc.max():.2f}")

BONAFIDE → LFCC shape: (20, 300) | min: -1909.55 | max: 406.99
SPOOF → LFCC shape: (20, 300) | min: -2578.90 | max: 507.78


In [ ]:
# Before normalization
lfcc_raw = lfcc_extraction(df[df['label']=='bonafide']['filepath'].iloc[0])
print(f"Before norm → min: {lfcc_raw.min():.2f} | max: {lfcc_raw.max():.2f} | mean: {lfcc_raw.mean():.2f}")

# Simple per-feature normalization (z-score)
mean = lfcc_raw.mean(axis=1, keepdims=True)  # mean per coefficient
std  = lfcc_raw.std(axis=1, keepdims=True)   # std per coefficient
lfcc_norm = (lfcc_raw - mean) / (std + 1e-8)

print(f"After norm  → min: {lfcc_norm.min():.2f} | max: {lfcc_norm.max():.2f} | mean: {lfcc_norm.mean():.2f}")

Before norm → min: -1909.55 | max: 406.99 | mean: -52.30
After norm  → min: -3.65 | max: 3.82 | mean: -0.00


**LFCC Extration:batch extraction**

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

feat_path = os.path.join(base_path, 'features/train')
os.makedirs(feat_path, exist_ok=True)

def process_single_file(row):
    out_path = os.path.join(feat_path, row['file_id'] + '.npy')

    # Skip if already extracted
    if os.path.exists(out_path):
        return row['file_id'], None
    try:
        lfcc = lfcc_extraction(row['filepath'])
        np.save(out_path, lfcc)
        return row['file_id'], None
    except Exception as e:
        return row['file_id'], str(e)

failed = []
rows = [row for _, row in df.iterrows()]

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(process_single_file, row): row for row in rows}
    for future in tqdm(as_completed(futures),
                       total=len(rows),
                       desc="Extracting LFCC"):
        file_id, error = future.result()
        if error:
            failed.append((file_id, error))

print(f" Saved : {len(os.listdir(feat_path))} files")
print(f"Failed: {len(failed)}")
if failed:
    print("Failed files:", failed[:5])


Extracting LFCC: 100%|██████████| 25380/25380 [01:27<00:00, 289.38it/s] 


 Saved : 25380 files
Failed: 0


**Normalizating statistics**

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def load_single_feat(row):
    feat_path_file = os.path.join(feat_path, row['file_id'] + '.npy')
    return np.load(feat_path_file)

# Parallel loading with 4 workers
rows = [row for _, row in df.iterrows()]
all_feats = [None] * len(rows)  # preallocate to preserve order

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(load_single_feat, row): i
               for i, row in enumerate(rows)}
    for future in tqdm(as_completed(futures),
                       total=len(rows),
                       desc="Loading features"):
        idx = futures[future]
        all_feats[idx] = future.result()

# Stack & compute stats
all_feats   = np.stack(all_feats)             # (25380, 20, 300)
global_mean = all_feats.mean(axis=(0,2), keepdims=True)  # (1, 20, 1)
global_std  = all_feats.std(axis=(0,2),  keepdims=True)  # (1, 20, 1)

print(f"Stacked shape: {all_feats.shape}")
print(f"Mean shape   : {global_mean.shape}")
print(f"Std  shape   : {global_std.shape}")
print(f"Mean range   : {global_mean.min():.2f} → {global_mean.max():.2f}")
print(f"Std  range   : {global_std.min():.2f} → {global_std.max():.2f}")

# Save to Drive
stats_dir = os.path.join(base_path, 'features')
np.save(f'{stats_dir}/global_mean.npy', global_mean)
np.save(f'{stats_dir}/global_std.npy',  global_std)
print("\n✅ Normalization stats saved to Drive!")

Loading features:   0%|          | 16/25380 [07:52<208:03:14, 29.53s/it]


KeyboardInterrupt: 

In [20]:
from sklearn.model_selection import train_test_split
train_df,val_df=train_test_split(
    df,
    train_size=0.9,
    random_state=42,
    stratify=df['binary_label']
)
print(f"Train size: {len(train_df)}")
print(f"Test size : {len(val_df)}")

Train size: 22842
Test size : 2538


In [13]:
stats_dir = os.path.join(base_path, 'features')
global_mean = np.load(f'{stats_dir}/global_mean.npy')
global_std  = np.load(f'{stats_dir}/global_std.npy')
print(f"global_mean shape : {global_mean.shape}")
print(f"global_std  shape : {global_std.shape}")
print(f"Mean range        : {global_mean.min():.2f} → {global_mean.max():.2f}")
print(f"Std  range        : {global_std.min():.2f} → {global_std.max():.2f}")

global_mean shape : (1, 20, 1)
global_std  shape : (1, 20, 1)
Mean range        : -1122.31 → 75.46
Std  range        : 14.34 → 721.95


In [14]:
class ASVspoofDataset(Dataset):
  def __init__(self,df,feat_path,global_mean,global_std):
    self.df=df.reset_index(drop=True)
    self.feat_path=feat_path
    self.global_mean=global_mean.squeeze()
    self.global_std=global_std.squeeze()
  def __len__(self):
    return len(self.df)
  def __getitem__(self, index):
    row=self.df.iloc[index]
    feat=np.load(os.path.join(self.feat_path,row['file_id']+'.npy'))
    feat=(feat-self.global_mean[:, None])/(self.global_std[:, None] + 1e-8)
    label=row['binary_label']

    feat_tensor=torch.FloatTensor(feat).unsqueeze(0)
    label_tensor=torch.tensor(label,dtype=torch.long)
    return feat_tensor,label_tensor


In [18]:
from torch.utils.data import WeightedRandomSampler
def get_sampler(df):
    n_bonafide = (df['binary_label'] == 0).sum()
    n_spoof    = (df['binary_label'] == 1).sum()
    w_bonafide = 1.0 / n_bonafide
    w_spoof    = 1.0 / n_spoof
    sample_weights = df['binary_label'].map(
        {0: w_bonafide, 1: w_spoof}
    ).values
    print(f"Bonafide weight : {w_bonafide:.6f}")
    print(f"Spoof weight    : {w_spoof:.6f}")
    print(f"Min weight      : {sample_weights.min():.6f}")
    return WeightedRandomSampler(
        weights     = torch.FloatTensor(sample_weights),
        num_samples = len(sample_weights),
        replacement = True
    )


In [19]:
batch_size    = 32
train_dataset = ASVspoofDataset(train_df, feat_path, global_mean, global_std)
val_dataset   = ASVspoofDataset(val_df,   feat_path, global_mean, global_std)
train_sampler = get_sampler(train_df)

Train_loader = DataLoader(
    train_dataset,
    batch_size  = batch_size,
    sampler     = train_sampler,
    num_workers = 2,
    pin_memory  = False
)
Val_loader = DataLoader(
    val_dataset,
    batch_size  = batch_size,
    shuffle     = False,
    num_workers = 2,
    pin_memory  = False
)

print(f"\n Train batches: {len(Train_loader)}")
print(f" Val batches  : {len(Val_loader)}")

# Verify one batch
batch_feat, batch_label = next(iter(Train_loader))
print(f"\nBatch feature shape : {batch_feat.shape}")
print(f"Batch label shape   : {batch_label.shape}")
print(f"Feature range       : {batch_feat.min():.2f} → {batch_feat.max():.2f}")
print(f"Labels in batch     : {batch_label.unique()}")
print(f"Label distribution  : {batch_label.bincount()}")

Bonafide weight : 0.000431
Spoof weight    : 0.000049
Min weight      : 0.000049

 Train batches: 714
 Val batches  : 80

Batch feature shape : torch.Size([32, 1, 20, 300])
Batch label shape   : torch.Size([32])
Feature range       : -6.02 → 5.91
Labels in batch     : tensor([0, 1])
Label distribution  : tensor([14, 18])


In [23]:

import json
os.makedirs(f'{base_path}/splits', exist_ok=True)

train_df.to_csv(f'{base_path}/splits/train_df.csv', index=False)
val_df.to_csv(f'{base_path}/splits/val_df.csv',     index=False)
print("DataFrames saved!")


handover = {
    'total_samples'  : len(df),
    'train_samples'  : len(train_df),
    'val_samples'    : len(val_df),
    'feature_shape'  : '(20, 300)',
    'n_lfcc'         : 20,
    'n_filter'       : 70,
    'n_fft'          : 512,
    'win_length'     : 400,
    'hop_length'     : 160,
    'sr'             : 16000,
    'max_frames'     : 300,
    'batch_size'     : 32,
    'label_0'        : 'bonafide',
    'label_1'        : 'spoof',
    'feat_dir'       : feat_path,
    'stats_dir'      : stats_dir,
    'splits_dir'     : f'{base_path}/splits'
}


os.makedirs(f'{base_path}/splits', exist_ok=True)
with open(f'{base_path}/splits/handover.json', 'w') as f:
    json.dump(handover, f, indent=4)
print("Handover config saved!")

DataFrames saved!
Handover config saved!


**Quick start cell: gift for u**

In [24]:

import os, json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')
base_path  = '/content/drive/MyDrive/ForensicAudio'
feat_path  = f'{base_path}/features/train'
stats_dir  = f'{base_path}/features'
splits_dir = f'{base_path}/splits'

# 2. Load config
with open(f'{splits_dir}/handover.json') as f:
    config = json.load(f)
print("Config loaded:", json.dumps(config, indent=2))

# 3. Load splits
train_df = pd.read_csv(f'{splits_dir}/train_df.csv')
val_df   = pd.read_csv(f'{splits_dir}/val_df.csv')
print(f"\nTrain : {len(train_df)} samples")
print(f"Val   : {len(val_df)} samples")

# 4. Load normalization stats
global_mean = np.load(f'{stats_dir}/global_mean.npy')
global_std  = np.load(f'{stats_dir}/global_std.npy')
print(f"\nMean shape: {global_mean.shape}")
print(f"Std  shape: {global_std.shape}")

# 5. Dataset class
class ASVspoofDataset(Dataset):
    def __init__(self, df, feat_dir, global_mean, global_std):
        self.df          = df.reset_index(drop=True)
        self.feat_dir    = feat_dir
        self.global_mean = global_mean.squeeze()
        self.global_std  = global_std.squeeze()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        feat = np.load(os.path.join(
                   self.feat_dir, row['file_id'] + '.npy'))
        feat = (feat - self.global_mean[:, None]) / \
               (self.global_std[:, None] + 1e-8)
        feat_tensor  = torch.FloatTensor(feat).unsqueeze(0)
        label_tensor = torch.tensor(row['binary_label'], dtype=torch.long)
        return feat_tensor, label_tensor

# 6. Sampler
def get_sampler(df):
    n_bonafide     = (df['binary_label'] == 0).sum()
    n_spoof        = (df['binary_label'] == 1).sum()
    sample_weights = df['binary_label'].map(
        {0: 1.0/n_bonafide, 1: 1.0/n_spoof}
    ).values
    return WeightedRandomSampler(
        weights     = torch.FloatTensor(sample_weights),
        num_samples = len(sample_weights),
        replacement = True
    )

# 7. DataLoaders
train_dataset = ASVspoofDataset(train_df, feat_path, global_mean, global_std)
val_dataset   = ASVspoofDataset(val_df,   feat_path, global_mean, global_std)

Train_loader  = DataLoader(
    train_dataset, batch_size=32,
    sampler=get_sampler(train_df), num_workers=2
)
Val_loader    = DataLoader(
    val_dataset, batch_size=32,
    shuffle=False, num_workers=2
)

# 8. Verify
batch_feat, batch_label = next(iter(Train_loader))
print(f"Batch shape     : {batch_feat.shape}")
print(f"Feature range   : {batch_feat.min():.2f} → {batch_feat.max():.2f}")
print(f"Label balance   : {batch_label.bincount()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Config loaded: {
  "total_samples": 25380,
  "train_samples": 22842,
  "val_samples": 2538,
  "feature_shape": "(20, 300)",
  "n_lfcc": 20,
  "n_filter": 70,
  "n_fft": 512,
  "win_length": 400,
  "hop_length": 160,
  "sr": 16000,
  "max_frames": 300,
  "batch_size": 32,
  "label_0": "bonafide",
  "label_1": "spoof",
  "feat_dir": "/content/drive/MyDrive/ForensicAudio/features/train",
  "stats_dir": "/content/drive/MyDrive/ForensicAudio/features",
  "splits_dir": "/content/drive/MyDrive/ForensicAudio/splits"
}

Train : 22842 samples
Val   : 2538 samples

Mean shape: (1, 20, 1)
Std  shape: (1, 20, 1)
Batch shape     : torch.Size([32, 1, 20, 300])
Feature range   : -5.71 → 5.71
Label balance   : tensor([17, 15])
